# MegaRAG: Multimodal Knowledge Graph-based RAG on Kaggle

This notebook reproduces **MegaRAG** on Kaggle using the bundled repository [`reproduce_MegaRAG`](https://github.com/linhnguyen15492/reproduce_MegaRAG.git), which contains the complete source code for **MegaRAG**, **LightRAG**, and **MinerU**.

MegaRAG enables **global visual question answering** on documents by constructing a **Multimodal Knowledge Graph (MMKG)** combining graph-based reasoning with document page retrieval.

### Prerequisites
- **GPU Accelerator**: Tesla T4 / P100 or better (enable GPU in Kaggle settings)
- **Internet**: Turned ON in Kaggle settings
- **OpenAI API Key**: Added to Kaggle Secrets as `OPENAI_API_KEY` (or entered interactively)

**Paper**: [MegaRAG: Multimodal Graph-based Retrieval Augmented Generation (ACL 2026)](https://arxiv.org/abs/2512.20626)


## 1. System Setup & Clone Repository


In [ ]:
import os
import sys
import shutil
import subprocess
from pathlib import Path
import torch

# GPU and PyTorch optimization flags
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# Set base working directory (default Kaggle working directory is /kaggle/working)
if Path("/kaggle/working").exists():
    BASE_DIR = Path("/kaggle/working")
else:
    BASE_DIR = Path.cwd()

# Clone or locate the reproduce_MegaRAG repository
REPO_URL = "https://github.com/linhnguyen15492/reproduce_MegaRAG.git"
REPO_NAME = "reproduce_MegaRAG"

if (BASE_DIR / "MegaRAG").exists() and (BASE_DIR / "MinerU").exists():
    REPO_DIR = BASE_DIR
elif (BASE_DIR / REPO_NAME / "MegaRAG").exists():
    REPO_DIR = BASE_DIR / REPO_NAME
else:
    print(f"Cloning repository {REPO_URL} into {BASE_DIR}...")
    subprocess.run(f"git clone {REPO_URL}", shell=True, check=True, cwd=BASE_DIR)
    REPO_DIR = BASE_DIR / REPO_NAME

os.chdir(REPO_DIR)
print(f"✓ Working directory: {REPO_DIR}")

# Set component paths from the repository
mineru_dir = REPO_DIR / "MinerU"
megarag_dir = REPO_DIR / "MegaRAG"
lightrag_dir = REPO_DIR / "LightRAG"


## 2. Install Required Dependencies & Local Packages


In [ ]:
import subprocess
import sys

# 1. Install base dependencies for MinerU, LightRAG, and MegaRAG
required_packages = [
    "pyopenssl>=24.0.0",
    "cryptography>=42.0.0",
    "transformers>=4.49.0,<5.0.0",
    "pillow>=10.2.0,<11.0.0",
    "PyMuPDF==1.24.14",
    "pdfminer.six==20231228",
    "pypdfium2",
    "rapid_table==1.0.3",
    "loguru",
    "boto3",
    "timm",
    "einops",
    "openai>=1.50.0,<2.0.0",
    "accelerate>=0.30.0,<2.0.0",
    "beautifulsoup4>=4.12.0",
    "opencv-python-headless",
    "ultralytics",
    "doclayout-yolo",
    "ftfy",
    "dill",
    "shapely",
    "pyclipper",
    "tiktoken",
    "huggingface_hub",
    "matplotlib",
    "rich",
    "pyyaml",
    "networkx",
]

print("Installing dependencies...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-warn-conflicts"] + required_packages)

# 2. Install bundled packages from repository in editable mode
print("Installing MinerU from repository...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(mineru_dir)])

print("Installing LightRAG from repository...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(lightrag_dir)])

print("Installing MegaRAG from repository...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(megarag_dir), "--no-deps"])

print("✓ All dependencies and packages installed successfully!")


## 3. Setup MinerU Models & Configuration


In [ ]:
import os
import shutil
import json as json_lib
from pathlib import Path
import torch
from huggingface_hub import snapshot_download

print("Downloading MinerU model weights from HuggingFace...")

# 1. Download PDF-Extract-Kit models
pdf_extract_kit_path = snapshot_download(
    repo_id="opendatalab/PDF-Extract-Kit-1.0",
    allow_patterns=["models/*"],
)
models_dir = Path(pdf_extract_kit_path) / "models"
print(f"✓ PDF-Extract-Kit models: {models_dir}")

# 2. Download LayoutReader model
layoutreader_path = snapshot_download(
    repo_id="hantian/layoutreader",
)
layoutreader_model_dir = Path(layoutreader_path)
print(f"✓ LayoutReader model: {layoutreader_model_dir}")

# 3. Setup OCR model aliases for compatibility
ocr_models_dir = models_dir / "OCR" / "paddleocr_torch"
if ocr_models_dir.exists():
    v5_det = ocr_models_dir / "ch_PP-OCRv5_det_infer.pth"
    v3_det = ocr_models_dir / "ch_PP-OCRv3_det_infer.pth"
    v4_det = ocr_models_dir / "ch_PP-OCRv4_det_infer.pth"
    if v5_det.exists():
        if not v3_det.exists():
            shutil.copy2(v5_det, v3_det)
        if not v4_det.exists():
            shutil.copy2(v5_det, v4_det)
    
    ppocr_dict = ocr_models_dir / "ppocr_keys_v1.txt"
    en_dict = ocr_models_dir / "en_dict.txt"
    if ppocr_dict.exists() and not en_dict.exists():
        shutil.copy2(ppocr_dict, en_dict)

# 4. Generate magic-pdf.json / mineru.json configuration
device_mode = "cuda" if torch.cuda.is_available() else "cpu"
config_data = {
    "models-dir": str(models_dir),
    "device-mode": device_mode,
    "layoutreader-model-dir": str(layoutreader_model_dir),
    "layout-config": {
        "model": "doclayout_yolo"
    },
    "latex-delimiter-config": {
        "display": {"left": "$$", "right": "$$"},
        "inline": {"left": "$", "right": "$"},
    },
}

config_paths = [
    Path.home() / "magic-pdf.json",
    Path("/root/magic-pdf.json"),
    Path.home() / "mineru.json",
    Path("/root/mineru.json"),
    mineru_dir / "magic-pdf.json",
    mineru_dir / "mineru.json",
]

for cfg_path in config_paths:
    try:
        cfg_path.parent.mkdir(parents=True, exist_ok=True)
        with open(cfg_path, "w", encoding="utf-8") as f:
            json_lib.dump(config_data, f, indent=4)
    except Exception:
        pass

print(f"✓ MinerU configuration generated successfully (device-mode: '{device_mode}')")


## 4. Setup API Keys & Environment Variables


In [ ]:
import os
import sys
import shutil
import getpass
from pathlib import Path

# Retrieve OpenAI API Key from Kaggle Secrets, Environment, or User Input
openai_api_key = os.environ.get("OPENAI_API_KEY")

if not openai_api_key:
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        openai_api_key = user_secrets.get_secret("OPENAI_API_KEY")
        print("✓ OpenAI API Key loaded from Kaggle Secrets.")
    except Exception:
        openai_api_key = None

if not openai_api_key:
    openai_api_key = getpass.getpass("Enter your OpenAI API Key: ")

os.environ["OPENAI_API_KEY"] = openai_api_key

# Update PATH with Python and MinerU bin locations
py_bin_dir = str(Path(sys.executable).parent)
extra_paths = [
    py_bin_dir,
    "/root/.local/bin",
    str(Path.home() / ".local" / "bin"),
    "/usr/local/bin",
    "/opt/conda/bin",
]
for p in extra_paths:
    if p not in os.environ.get("PATH", ""):
        os.environ["PATH"] = f"{p}:{os.environ.get('PATH', '')}"

# Write env.sh in MegaRAG directory as required by MegaRAG scripts
mineru_bin = shutil.which("magic-pdf") or shutil.which("mineru") or py_bin_dir
mineru_bin_dir = str(Path(mineru_bin).parent) if mineru_bin else str(Path(sys.executable).parent)

env_sh_content = f"""#!/usr/bin/env bash
export OPENAI_API_KEY="{openai_api_key}"
export MINERU_PATH="{mineru_bin_dir}"
"""

env_file = megarag_dir / "env.sh"
with open(env_file, "w", encoding="utf-8") as f:
    f.write(env_sh_content)

print(f"✓ Configured {env_file}")
print(f"✓ magic-pdf CLI: {shutil.which('magic-pdf') or 'Available via python module'}")


## 5. Prepare Example Data (`world_history_tiny`)

Load `World_History_Volume_1.pdf` and `queries.txt` from `/kaggle/input/datasets` (or `/kaggle/input/`) into the `egs/world_history_tiny/data/` folder.


In [ ]:
import os
import shutil
from pathlib import Path

example_dir = megarag_dir / "egs" / "world_history_tiny"
data_dir = example_dir / "data"
data_dir.mkdir(parents=True, exist_ok=True)

pdf_file = data_dir / "World_History_Volume_1.pdf"
queries_file = data_dir / "queries.txt"

# 1. Search for dataset files in /kaggle/input/datasets or /kaggle/input
kaggle_input = Path("/kaggle/input")
search_roots = [
    kaggle_input / "datasets",
    kaggle_input,
]

pdf_source = None
queries_source = None

for root in search_roots:
    if root.exists():
        # Look for PDF file
        if not pdf_source:
            pdf_matches = list(root.glob("**/World_History_Volume_1.pdf"))
            if pdf_matches:
                pdf_source = pdf_matches[0]
        # Look for queries file
        if not queries_source:
            query_matches = list(root.glob("**/queries.txt"))
            if query_matches:
                queries_source = query_matches[0]

# Copy PDF from Kaggle Input to data directory
if pdf_source and pdf_source.exists():
    shutil.copy2(pdf_source, pdf_file)
    print(f"✓ Found and copied PDF from: {pdf_source}")
elif pdf_file.exists():
    print(f"✓ PDF already present in data directory: {pdf_file}")
else:
    print(f"⚠️ PDF file 'World_History_Volume_1.pdf' not found in /kaggle/input/datasets!")
    print(f"   Please make sure the dataset is added to the Kaggle notebook input.")

# Copy or generate queries.txt
if queries_source and queries_source.exists():
    shutil.copy2(queries_source, queries_file)
    print(f"✓ Found and copied queries from: {queries_source}")
elif not queries_file.exists():
    sample_queries = """- Question 1: What is the Byzantine Empire?
- Question 2: When did the Roman Empire fall?
- Question 3: Who was Alexander the Great?
- Question 4: What was the Silk Road?
- Question 5: Describe ancient Egyptian civilization.
"""
    with open(queries_file, "w", encoding="utf-8") as f:
        f.write(sample_queries.strip() + "\n")
    print(f"✓ Created benchmark queries file at: {queries_file}")
else:
    print(f"✓ Queries file ready: {queries_file}")

if pdf_file.exists():
    print(f"✓ Input PDF ready: {pdf_file} ({pdf_file.stat().st_size / (1024 * 1024):.2f} MB)")


## 6. Build Multimodal Knowledge Graph (MMKG)

Following Step 3 of MegaRAG README (`run_build_mmkg.sh`):
1. **Parse PDF**: Extract text, layout, tables, and images using MinerU (`magic-pdf`).
2. **PDF to Images**: Convert PDF pages to images using `pdf2img.py`.
3. **Build Page Assets**: Merge page content and visual assets with `build_page_assets.py`.
4. **Construct MMKG**: Build the graph and multimodal embeddings with `construct_mmkg.py`.


In [ ]:
import os
import sys
import time
import shutil
import subprocess
from pathlib import Path

os.chdir(example_dir)
print(f"Working directory: {example_dir}")

pdf_path = data_dir / "World_History_Volume_1.pdf"
pdf_name = pdf_path.stem
dumps_dir = example_dir / "dumps"
dumps_dir.mkdir(parents=True, exist_ok=True)
exp_dir = example_dir / "exp" / pdf_name
exp_dir.mkdir(parents=True, exist_ok=True)
config_file = example_dir / "conf" / "addon_params.yaml"

# Step 1: Parse PDF using MinerU (pages 0-9 for quickstart)
print("\n--- [Step 1] Parsing PDF with MinerU ---")
magic_pdf_bin = shutil.which("magic-pdf") or f"{sys.executable} -m magic_pdf.cli.magicpdf"
cmd_parse = f"{magic_pdf_bin} -p {pdf_path} -o {dumps_dir} -l en -e 9"
print(f"Running: {cmd_parse}")
subprocess.run(cmd_parse, shell=True, check=True)

# Step 2: Convert PDF pages to images
print("\n--- [Step 2] Converting PDF pages to images ---")
pdf2img_py = megarag_dir / "egs" / "utils" / "pdf2img.py"
page_images_dir = dumps_dir / pdf_name / "auto" / "page_images"
page_images_dir.mkdir(parents=True, exist_ok=True)

cmd_img = f"{sys.executable} {pdf2img_py} {pdf_path} {page_images_dir} --dpi 150 --jpeg --end-page 10 --jobs 4"
print(f"Running: {cmd_img}")
subprocess.run(cmd_img, shell=True, check=True)

# Step 3: Build Page Assets Manifest
print("\n--- [Step 3] Building Page Assets Manifest ---")
build_assets_py = megarag_dir / "egs" / "utils" / "build_page_assets.py"
auto_dir = dumps_dir / pdf_name / "auto"
page_manifest = dumps_dir / pdf_name / "pages_content.json"

cmd_assets = f"{sys.executable} {build_assets_py} --working-dir {auto_dir} --output {page_manifest}"
print(f"Running: {cmd_assets}")
subprocess.run(cmd_assets, shell=True, check=True)

# Step 4: Construct Multimodal Knowledge Graph
print("\n--- [Step 4] Constructing Multimodal Knowledge Graph ---")
construct_mmkg_py = megarag_dir / "egs" / "utils" / "construct_mmkg.py"

cmd_mmkg = f"{sys.executable} {construct_mmkg_py} --config-file {config_file} --working-dir {exp_dir} --input-dir {page_manifest}"
print(f"Running: {cmd_mmkg}")
t0 = time.time()
subprocess.run(cmd_mmkg, shell=True, check=True)
print(f"\n✓ MMKG Construction completed in {time.time() - t0:.1f} seconds!")


## 7. Query with MegaRAG

Following Step 4 of MegaRAG README (`run_quering.sh`): Query the Multimodal Knowledge Graph using `query_mmkg.py`.


In [ ]:
import os
import sys
import time
import subprocess
from pathlib import Path

query_script = megarag_dir / "egs" / "utils" / "query_mmkg.py"
results_dir = exp_dir / "results"
results_dir.mkdir(parents=True, exist_ok=True)
results_file = results_dir / "results.json"

cmd_query = f"""{sys.executable} {query_script} \
    --config-file {config_file} \
    --working-dir {exp_dir} \
    --input-queries {queries_file} \
    --output-file {results_file} \
    --concurrency 4
"""

print(f"Running queries:\n{cmd_query}\n")
t0 = time.time()
subprocess.run(cmd_query, shell=True, check=True)
print(f"\n✓ Querying completed in {time.time() - t0:.1f} seconds!")


## 8. View Results & Knowledge Graph Analysis


In [ ]:
import json
from pathlib import Path
import networkx as nx

# 1. Display Query Results
if results_file.exists():
    with open(results_file, "r", encoding="utf-8") as f:
        results = json.load(f)

    print("=" * 80)
    print("MEGARAG QUERY RESULTS")
    print("=" * 80)

    if isinstance(results, list):
        for i, res in enumerate(results, 1):
            print(f"\n{'─' * 80}")
            print(f"Query {i}: {res.get('query', 'N/A')}")
            print(f"{'─' * 80}")
            if "answer" in res:
                print(f"Answer:\n{res['answer']}\n")
            if "retrieved_context" in res:
                print(f"Retrieved Context:\n{res['retrieved_context'][:300]}...\n")
    elif isinstance(results, dict):
        for query, ans in results.items():
            print(f"\n{'─' * 80}")
            print(f"Query: {query}")
            print(f"{'─' * 80}")
            answer_text = ans.get("answer", ans) if isinstance(ans, dict) else ans
            print(f"Answer:\n{answer_text}\n")
    print("=" * 80)

# 2. Knowledge Graph Analysis
graphml_files = list(exp_dir.glob("**/*.graphml"))
if graphml_files:
    G = nx.read_graphml(graphml_files[0])
    print(f"\nKnowledge Graph Overview ({graphml_files[0].name}):")
    print(f"  • Total Entities (Nodes): {G.number_of_nodes()}")
    print(f"  • Total Relationships (Edges): {G.number_of_edges()}")
    
    degrees = dict(G.degree())
    top_nodes = sorted(degrees.items(), key=lambda x: x[1], reverse=True)[:5]
    print("\nTop 5 Most Connected Entities:")
    for rank, (node, deg) in enumerate(top_nodes, 1):
        entity_type = G.nodes[node].get("entity_type", "entity")
        print(f"  {rank}. [{entity_type}] {node} ({deg} connections)")
